## Cellular protein embeddingsssss :)

In [1]:
import os
import sys
import ast
import json
import time
import shutil
import pickle
import argparse

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer

PROJECT_DIR = "/Users/dylanwells/popDMS/esmDMS"
sys.path.insert(0, PROJECT_DIR)

from esmdmsfunctions import CODON2AA

DATA_DIR = os.path.join(PROJECT_DIR, "data", "raw_data")
HOME_SEQ_FOLDER = os.path.join(PROJECT_DIR, "data", "sequence_data")

In [2]:
# Sample file format:

data_dir = "/Users/dylanwells/popDMS/esmDMS/data/raw_data/"

tpor_filepath = data_dir + "TpoR_nucleotide_counts.csv"
tpor_reference_filepath = data_dir + "TpoR_reference_sequence.dat"

'''
accession,hgvs_nt,hgvs_splice,hgvs_pro,Replicate_A_c_0,Replicate_A_c_1,Replicate_B_c_0,Replicate_B_c_1,Replicate_C_c_0,Replicate_C_c_1,Replicate_D_c_0,Replicate_D_c_1,Replicate_E_c_0,Replicate_E_c_1,Replicate_F_c_0,Replicate_F_c_1
urn:mavedb:00000043-a-1#1,c.93T>G,NA,p.Phe31Leu,5.0,19.0,5.0,7.0,5.0,22.0,5.0,18.0,5.0,30.0,5.0,20.0
urn:mavedb:00000043-a-1#2,c.93T>A,NA,p.Phe31Leu,71.0,195.0,71.0,104.0,71.0,209.0,71.0,192.0,71.0,232.0,71.0,208.0
urn:mavedb:00000043-a-1#3,c.92T>G,NA,p.Phe31Cys,34.0,51.0,34.0,39.0,34.0,68.0,34.0,51.0,34.0,65.0,34.0,84.0
urn:mavedb:00000043-a-1#4,c.92T>C,NA,p.Phe31Ser,70.0,49.0,70.0,35.0,70.0,55.0,70.0,62.0,70.0,40.0,70.0,55.0
urn:mavedb:00000043-a-1#5,c.92T>A,NA,p.Phe31Tyr,75.0,54.0,75.0,36.0,75.0,62.0,75.0,57.0,75.0,64.0,75.0,52.0
urn:mavedb:00000043-a-1#6,c.91T>G,NA,p.Phe31Val,111.0,99.0,111.0,57.0,111.0,93.0,111.0,146.0,111.0,99.0,111.0,137.0
urn:mavedb:00000043-a-1#7,c.91T>C,NA,p.Phe31Leu,45.0,33.0,45.0,18.0,45.0,30.0,45.0,30.0,45.0,29.0,45.0,28.0
urn:mavedb:00000043-a-1#8,c.91T>A,NA,p.Phe31Ile,87.0,85.0,87.0,22.0,87.0,110.0,87.0,77.0,87.0,93.0,87.0,83.0
'''

'\naccession,hgvs_nt,hgvs_splice,hgvs_pro,Replicate_A_c_0,Replicate_A_c_1,Replicate_B_c_0,Replicate_B_c_1,Replicate_C_c_0,Replicate_C_c_1,Replicate_D_c_0,Replicate_D_c_1,Replicate_E_c_0,Replicate_E_c_1,Replicate_F_c_0,Replicate_F_c_1\nurn:mavedb:00000043-a-1#1,c.93T>G,NA,p.Phe31Leu,5.0,19.0,5.0,7.0,5.0,22.0,5.0,18.0,5.0,30.0,5.0,20.0\nurn:mavedb:00000043-a-1#2,c.93T>A,NA,p.Phe31Leu,71.0,195.0,71.0,104.0,71.0,209.0,71.0,192.0,71.0,232.0,71.0,208.0\nurn:mavedb:00000043-a-1#3,c.92T>G,NA,p.Phe31Cys,34.0,51.0,34.0,39.0,34.0,68.0,34.0,51.0,34.0,65.0,34.0,84.0\nurn:mavedb:00000043-a-1#4,c.92T>C,NA,p.Phe31Ser,70.0,49.0,70.0,35.0,70.0,55.0,70.0,62.0,70.0,40.0,70.0,55.0\nurn:mavedb:00000043-a-1#5,c.92T>A,NA,p.Phe31Tyr,75.0,54.0,75.0,36.0,75.0,62.0,75.0,57.0,75.0,64.0,75.0,52.0\nurn:mavedb:00000043-a-1#6,c.91T>G,NA,p.Phe31Val,111.0,99.0,111.0,57.0,111.0,93.0,111.0,146.0,111.0,99.0,111.0,137.0\nurn:mavedb:00000043-a-1#7,c.91T>C,NA,p.Phe31Leu,45.0,33.0,45.0,18.0,45.0,30.0,45.0,30.0,45.0,29.0,45.0,2

## Data Processing — MaveDB Format

The raw TpoR (and similar MaveDB) files have a different layout than the codon-count CSVs used in `embed_sequences.py`.  Key differences:

- Mutation identity comes from the `hgvs_nt` column. Entries can be single substitutions (`c.90G>C`) or multiple nucleotide changes within the same or different codons (`c.[8C>T;9C>G]`).
- The reference file is a **nucleotide** CDS sequence. All substitutions are applied to it and the result is translated codon-by-codon via `CODON2AA`.
- Only variants that change **exactly one amino acid** relative to the reference are kept (single-mutant filter). Rows where substitutions span two or more codons (double mutants) are discarded.
- Pre- and post-selection counts are interleaved per replicate: `Replicate_A_c_0` (pre), `Replicate_A_c_1` (post), `Replicate_B_c_0`, …

The functions below convert this format into the same `PreNums / PostNums / ProteinSequence` DataFrame that `embed_sequences.py` produces.

In [ ]:
import re
import io

# Regex for a single nucleotide substitution token, e.g. "90G>C" or "8C>T"
_SNV_RE = re.compile(r'^(\d+)([ACGTacgt])>([ACGTacgt])$')


def parse_hgvs_nt(hgvs_str):
    """
    Parse an HGVS nucleotide string into a list of (pos_0indexed, ref_nuc, alt_nuc) tuples.

    Handles single and multi-substitution entries:
        'c.90G>C'        → [(89, 'G', 'C')]
        'c.[8C>T;9C>G]'  → [(7, 'C', 'T'), (8, 'C', 'G')]

    Returns an empty list for anything that cannot be parsed as one or more
    plain single-nucleotide substitutions (indels, frameshifts, NaN, etc.).
    Any non-SNV token in a multi-change entry invalidates the whole row.
    '_wt' is handled separately in build_sequence_dataframe_mavedb.
    """
    if not isinstance(hgvs_str, str):
        return []
    hgvs_str = hgvs_str.strip()

    if hgvs_str.startswith('c.[') and hgvs_str.endswith(']'):
        tokens = hgvs_str[3:-1].split(';')
    elif hgvs_str.startswith('c.'):
        tokens = [hgvs_str[2:]]
    else:
        return []

    result = []
    for tok in tokens:
        m = _SNV_RE.match(tok.strip())
        if m is None:
            return []  # non-SNV token — discard entire row
        pos_1indexed = int(m.group(1))
        ref_nuc, alt_nuc = m.group(2).upper(), m.group(3).upper()
        if ref_nuc == alt_nuc:
            return []
        result.append((pos_1indexed - 1, ref_nuc, alt_nuc))  # 0-indexed

    return result


def get_reference_nuc_sequence(filepath):
    """
    Read a nucleotide CDS reference file (plain text or FASTA) and return the
    raw uppercase nucleotide string with whitespace stripped.
    """
    with open(filepath, 'r') as f:
        lines = f.readlines()
    return ''.join(
        line.strip() for line in lines if not line.startswith('>')
    ).upper()


def translate_nuc_sequence(nuc_seq):
    """Translate a nucleotide string to an amino-acid string using CODON2AA."""
    return ''.join(
        CODON2AA.get(nuc_seq[i:i + 3], 'X') for i in range(0, len(nuc_seq) - 2, 3)
    )


def apply_substitutions(ref_nuc_seq, substitutions):
    """
    Apply a list of (pos_0indexed, ref_nuc, alt_nuc) substitutions to the
    reference CDS and return the mutant nucleotide sequence.

    Returns None if any substitution position is out of range or the reference
    nucleotide at that position does not match the claimed ref_nuc.
    """
    nuc_list = list(ref_nuc_seq)
    for pos, ref_nuc, alt_nuc in substitutions:
        if pos >= len(nuc_list):
            return None
        if nuc_list[pos] != ref_nuc:
            return None  # reference mismatch
        nuc_list[pos] = alt_nuc
    return ''.join(nuc_list)


def _extract_replicate_columns(df_columns):
    """
    Detect replicate pre/post count column pairs.

    Expects names like 'Replicate_A_c_0' (pre) and 'Replicate_A_c_1' (post).
    Returns two parallel lists (pre_cols, post_cols) in order of first appearance.
    """
    seen = {}
    for col in df_columns:
        m = re.match(r'^(Replicate_\w+)_c_(\d+)$', col)
        if m:
            rep_label, timepoint = m.group(1), int(m.group(2))
            seen.setdefault(rep_label, {})[timepoint] = col

    pre_cols, post_cols = [], []
    for rep_label, tp_map in seen.items():
        if 0 in tp_map and 1 in tp_map:
            pre_cols.append(tp_map[0])
            post_cols.append(tp_map[1])
    return pre_cols, post_cols


def build_sequence_dataframe_mavedb(csv_filepath, ref_nuc_seq, skip_stop_codons=True):
    """
    Load a MaveDB-format CSV and return a DataFrame with columns:
        PreNums          – list of pre-selection counts (one per replicate)
        PostNums         – list of post-selection counts (one per replicate)
        ProteinSequence  – full protein sequence (reference or mutant)

    '_wt' rows are included as the reference protein sequence.
    For all other rows, nucleotide substitutions from hgvs_nt are applied to
    ref_nuc_seq and the result is translated.

    Cannot use pd.read_csv(comment='#') because '#' appears inside MaveDB
    accession values (e.g. 'urn:mavedb:00000043-a-1#1'), which would corrupt
    every data row. Instead, file-level comment lines are stripped manually.
    """
    with open(csv_filepath) as f:
        content = ''.join(line for line in f if not line.startswith('#'))
    df = pd.read_csv(io.StringIO(content))

    pre_cols, post_cols = _extract_replicate_columns(df.columns.tolist())
    if not pre_cols:
        raise ValueError(
            "No replicate count columns found. Expected names like 'Replicate_A_c_0'."
        )

    ref_aa_seq = translate_nuc_sequence(ref_nuc_seq)

    records = []
    for _, row in df.iterrows():
        hgvs_nt = row.get('hgvs_nt', None)

        if hgvs_nt == '_wt':
            aa_seq = ref_aa_seq
        else:
            substitutions = parse_hgvs_nt(hgvs_nt)
            if not substitutions:
                continue
            mut_nuc_seq = apply_substitutions(ref_nuc_seq, substitutions)
            if mut_nuc_seq is None:
                continue
            aa_seq = translate_nuc_sequence(mut_nuc_seq)

        records.append({
            'PreNums':  [int(row[c]) if pd.notna(row[c]) else 0 for c in pre_cols],
            'PostNums': [int(row[c]) if pd.notna(row[c]) else 0 for c in post_cols],
            'ProteinSequence': aa_seq,
        })

    result = pd.DataFrame(records, columns=['PreNums', 'PostNums', 'ProteinSequence'])

    # Aggregate rows that map to the same protein sequence (e.g. two synonymous
    # nucleotide changes giving the same AA substitution, or multiple _wt rows)
    if not result.empty:
        result = result.groupby('ProteinSequence', as_index=False).agg({
            'PreNums':  lambda x: [sum(v) for v in zip(*x)],
            'PostNums': lambda x: [sum(v) for v in zip(*x)],
        })

    return result


In [12]:
# --- Build the sequence DataFrame for TpoR ---
#ref_nuc_seq = get_reference_nuc_sequence(tpor_reference_filepath)
#ref_aa_seq  = translate_nuc_sequence(ref_nuc_seq)
#print(f"Reference CDS length : {len(ref_nuc_seq)} nt  ({len(ref_aa_seq)} aa)")
#print(f"First 20 aa          : {ref_aa_seq[:20]}")

seq_df = build_sequence_dataframe_mavedb(tpor_filepath, ref_nuc_seq)
#print(f"\nUnique mutant sequences : {len(seq_df)}")
#print(f"Replicates per row      : {len(seq_df['PreNums'].iloc[0])}")

[(92, 'T', 'G')]
[(92, 'T', 'A')]
[(91, 'T', 'G')]
[(91, 'T', 'C')]
[(91, 'T', 'A')]
[(90, 'T', 'G')]
[(90, 'T', 'C')]
[(90, 'T', 'A')]
[(89, 'G', 'T')]
[(89, 'G', 'C')]
[(7, 'C', 'T'), (8, 'C', 'G')]
[(7, 'C', 'T'), (8, 'C', 'T')]
[(7, 'C', 'T'), (8, 'C', 'A')]
[(7, 'C', 'T'), (85, 'G', 'T')]
[(7, 'C', 'T'), (55, 'G', 'A')]
[(7, 'C', 'T')]
[(7, 'C', 'G'), (8, 'C', 'T')]
[(7, 'C', 'G'), (85, 'G', 'T'), (86, 'G', 'A')]
[(7, 'C', 'G'), (85, 'G', 'T')]
[(7, 'C', 'G')]
[(7, 'C', 'G'), (8, 'C', 'G')]
[(7, 'C', 'G'), (8, 'C', 'A')]
[(7, 'C', 'A'), (8, 'C', 'G')]
[(7, 'C', 'A'), (8, 'C', 'A')]
[(7, 'C', 'A'), (8, 'C', 'T'), (85, 'G', 'T')]
[(7, 'C', 'A'), (8, 'C', 'T')]
[(7, 'C', 'A'), (85, 'G', 'T'), (86, 'G', 'A')]
[(7, 'C', 'A'), (85, 'G', 'T')]
[(7, 'C', 'A'), (85, 'G', 'C')]
[(7, 'C', 'A'), (85, 'G', 'A'), (86, 'G', 'C')]
[(7, 'C', 'A'), (84, 'T', 'G'), (85, 'G', 'C')]
[(7, 'C', 'A'), (84, 'T', 'A'), (86, 'G', 'A')]
[(7, 'C', 'A'), (55, 'G', 'A'), (56, 'C', 'T')]
[(7, 'C', 'A'), (55, 'G'

In [5]:
seq_df

,ProteinSequence,PreNums,PostNums
0,AETAWISLVTALHLVLGLSAVLGLLLLRWQF,"[38, 38, 38, 38, 38, 38]","[101, 22, 25, 49, 42, 73]"
1,IETAWISLVTALHLVLGLNAVLGLLLLRWQF,"[1, 1, 1, 1, 1, 1]","[4, 4, 6, 5, 6, 8]"
2,IETAWISLVTALHLVLGLSAVLGLLLLRKQF,"[1, 1, 1, 1, 1, 1]","[6, 3, 7, 6, 5, 6]"
3,IETAWISLVTALHLVLGLSAVLGLLLLRWQF,"[54, 54, 54, 54, 54, 54]","[36, 28, 27, 59, 39, 82]"
4,NETAWISLVTALHLSLGLSAVLGLLLLRWQF,"[1, 1, 1, 1, 1, 1]","[7, 6, 10, 9, 18, 11]"
...,...,...,...
1127,TVTAWISLVTALHLVLGLNAVLGLLLLRWQF,"[1, 1, 1, 1, 1, 1]","[12, 12, 19, 29, 17, 64]"
1128,TVTAWISLVTALHLVLGLSAVLGLLLLRSQF,"[4, 4, 4, 4, 4, 4]","[10, 4, 6, 11, 5, 21]"
1129,TVTAWISLVTALHLVLGLSAVLGLLLLRWQF,"[4034, 4034, 4034, 4034, 4034, 4034]","[603, 395, 1079, 1347, 741, 3300]"
1130,TWTAWISLVTALHLVLGLSAVLGLLLLRWQF,"[1390, 1390, 1390, 1390, 1390, 1390]","[3384, 52, 376, 201, 1188, 178]"


In [6]:
sys.path.insert(0, f"{PROJECT_DIR}/embedding_scripts")
                
from embed_sequences import embed_dataframe

test_embed_df = embed_dataframe(seq_df, esm_model="facebook/esm2_t33_650M_UR50D")
test_embed_df

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  [5/1132] elapsed 0.7 min, est. remaining 158.5 min
  Memory usage: 1091.0 MB
  [10/1132] elapsed 1.2 min, est. remaining 139.1 min
  Memory usage: 1039.4 MB
  [15/1132] elapsed 1.8 min, est. remaining 132.9 min
  Memory usage: 1114.5 MB
  [20/1132] elapsed 2.4 min, est. remaining 134.3 min
  Memory usage: 848.6 MB
  [25/1132] elapsed 3.0 min, est. remaining 134.4 min
  Memory usage: 611.1 MB
  [30/1132] elapsed 3.6 min, est. remaining 133.7 min
  Memory usage: 641.3 MB
  [35/1132] elapsed 4.2 min, est. remaining 132.1 min
  Memory usage: 1108.5 MB
  [40/1132] elapsed 4.8 min, est. remaining 130.2 min
  Memory usage: 874.5 MB
  [45/1132] elapsed 5.4 min, est. remaining 129.5 min
  Memory usage: 1080.0 MB


KeyboardInterrupt: 

In [ ]:
import psutil, os
print(f"Memory: {psutil.Process(os.getpid()).memory_info().rss / 1024**2:.1f} MB")

Memory: 169.4 MB


In [7]:
seq_df

,ProteinSequence,PreNums,PostNums
0,AETAWISLVTALHLVLGLSAVLGLLLLRWQF,"[38, 38, 38, 38, 38, 38]","[101, 22, 25, 49, 42, 73]"
1,IETAWISLVTALHLVLGLNAVLGLLLLRWQF,"[1, 1, 1, 1, 1, 1]","[4, 4, 6, 5, 6, 8]"
2,IETAWISLVTALHLVLGLSAVLGLLLLRKQF,"[1, 1, 1, 1, 1, 1]","[6, 3, 7, 6, 5, 6]"
3,IETAWISLVTALHLVLGLSAVLGLLLLRWQF,"[54, 54, 54, 54, 54, 54]","[36, 28, 27, 59, 39, 82]"
4,NETAWISLVTALHLSLGLSAVLGLLLLRWQF,"[1, 1, 1, 1, 1, 1]","[7, 6, 10, 9, 18, 11]"
...,...,...,...
1127,TVTAWISLVTALHLVLGLNAVLGLLLLRWQF,"[1, 1, 1, 1, 1, 1]","[12, 12, 19, 29, 17, 64]"
1128,TVTAWISLVTALHLVLGLSAVLGLLLLRSQF,"[4, 4, 4, 4, 4, 4]","[10, 4, 6, 11, 5, 21]"
1129,TVTAWISLVTALHLVLGLSAVLGLLLLRWQF,"[4034, 4034, 4034, 4034, 4034, 4034]","[603, 395, 1079, 1347, 741, 3300]"
1130,TWTAWISLVTALHLVLGLSAVLGLLLLRWQF,"[1390, 1390, 1390, 1390, 1390, 1390]","[3384, 52, 376, 201, 1188, 178]"


In [8]:
seq_df['ProteinSequence'].iloc[0]

'AETAWISLVTALHLVLGLSAVLGLLLLRWQF'

In [9]:
len("ACCGAGACCGCCTGGATCTCCTTGGTGACCGCTCTGCATCTAGTGCTGGGCCTCAGCGCCGTCCTGGGCCTGCTGCTGCTGAGGTGGCAGTTT")

93

In [13]:
ref_nuc_seq = get_reference_nuc_sequence(tpor_reference_filepath)
ref_aa_seq  = translate_nuc_sequence(ref_nuc_seq)

# --- Wildtype ---
print("=== Wildtype reference sequence ===")
print(ref_aa_seq)
print(f"  ({len(ref_aa_seq)} aa)\n")

# --- Example mutants ---
print("=== Example mutant sequences (first 10) ===")
print(f"{'Substitution':<14}  {'Mutant sequence'}")
print("-" * 55)

n_shown = 0
for _, row in seq_df.iterrows():
    mut_seq = row['ProteinSequence']
    if mut_seq == ref_aa_seq:
        continue  # skip wildtype row

    diffs = [(i, ref_aa_seq[i], mut_seq[i])
             for i in range(min(len(ref_aa_seq), len(mut_seq)))
             if ref_aa_seq[i] != mut_seq[i]]

    label = ', '.join(f"{r}{i+1}{m}" for i, r, m in diffs)
    print(f"{label:<14}  {mut_seq}")

    n_shown += 1
    if n_shown >= 10:
        break


=== Wildtype reference sequence ===
TETAWISLVTALHLVLGLSAVLGLLLLRWQF
  (31 aa)

=== Example mutant sequences (first 10) ===
Substitution    Mutant sequence
-------------------------------------------------------
T1A             AETAWISLVTALHLVLGLSAVLGLLLLRWQF
T1I, S19N       IETAWISLVTALHLVLGLNAVLGLLLLRWQF
T1I, W29K       IETAWISLVTALHLVLGLSAVLGLLLLRKQF
T1I             IETAWISLVTALHLVLGLSAVLGLLLLRWQF
T1N, V15S       NETAWISLVTALHLSLGLSAVLGLLLLRWQF
T1N, S19N       NETAWISLVTALHLVLGLNAVLGLLLLRWQF
T1N, W29M       NETAWISLVTALHLVLGLSAVLGLLLLRMQF
T1N, W29Q       NETAWISLVTALHLVLGLSAVLGLLLLRQQF
T1N, W29R       NETAWISLVTALHLVLGLSAVLGLLLLRRQF
T1N, W29S       NETAWISLVTALHLVLGLSAVLGLLLLRSQF
